# P2P · document.marker.convert (V1 · interactive Colab)

Converts **one** PDF to Markdown with [Marker](https://github.com/datalab-to/marker) on a free Colab GPU and returns **one ZIP bundle** you download and validate offline with P2P (`marker_bundle_validator.py`).

**How to use (no typing needed):**
1. `Runtime → Change runtime type → T4 GPU` (free tier). Save.
2. `Runtime → Run all`.
3. When the upload button appears, pick **one non‑sensitive PDF**.
4. Wait. The ZIP downloads automatically at the end.

**What this notebook will NOT do:** ask for any token/credential, mount Drive, call a paid API, use an external LLM, or upgrade you to paid Colab. If no free GPU is assigned it **stops** and tells you.

**Large PDFs:** > `FULL_CONVERSION_MAX_PAGES` pages are converted as a deterministic *sample* first. Re‑run with `FORCE_FULL = True` only after the sample looks right.


In [ ]:
#@title 0 · Settings (defaults are fine)
FULL_CONVERSION_MAX_PAGES = 20   #@param {type:"integer"}
FORCE_FULL = False               #@param {type:"boolean"}
MIN_TEXT_CHARS_PER_PAGE = 40     # a page with fewer chars AND a raster image is treated as scanned -> OCR
BUNDLE_SCHEMA = "p2p-document-marker-convert-bundle/1"
CAPABILITY_ID = "document.marker.convert"
print("settings ok")

In [ ]:
#@title 1 · GPU check (fails closed — no paid upgrade)
import subprocess, shutil, sys
gpu_name = None
if shutil.which("nvidia-smi"):
    q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                       capture_output=True, text=True)
    if q.returncode == 0 and q.stdout.strip():
        gpu_name = q.stdout.strip().splitlines()[0]
if not gpu_name:
    print("=" * 70)
    print("STOP: no free GPU is assigned to this Colab runtime.")
    print("Fix: Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save,")
    print("then Runtime -> Run all again. If Colab says GPUs are unavailable, try later.")
    print("This notebook will NOT request a paid upgrade.")
    print("=" * 70)
    raise SystemExit("GPU_UNAVAILABLE")
print("GPU:", gpu_name)

In [ ]:
#@title 2 · Install Marker + PyMuPDF (quiet, ~2-4 min)
import os, sys, subprocess
os.environ["TORCH_DEVICE"] = "cuda"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "marker-pdf", "pymupdf"], check=True)
import importlib.metadata as im, torch
MARKER_VERSION = im.version("marker-pdf")
print("marker-pdf", MARKER_VERSION, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("GPU_UNAVAILABLE_TO_TORCH")

In [ ]:
#@title 3 · Upload ONE PDF
import hashlib, pathlib, shutil, time, datetime
from google.colab import files
WORK = pathlib.Path("/content/p2p_marker_run"); shutil.rmtree(WORK, ignore_errors=True)
(WORK / "source").mkdir(parents=True); (WORK / "output" / "images").mkdir(parents=True)
uploaded = files.upload()
pdfs = [n for n in uploaded if n.lower().endswith(".pdf")]
if len(pdfs) != 1:
    raise SystemExit(f"UPLOAD_EXACTLY_ONE_PDF (got {len(pdfs)})")
ORIGINAL_NAME = pdfs[0]
SOURCE_BYTES = uploaded[ORIGINAL_NAME]
if not SOURCE_BYTES.startswith(b"%PDF-"):
    raise SystemExit("NOT_A_PDF")
SOURCE_PATH = WORK / "source" / "original.pdf"; SOURCE_PATH.write_bytes(SOURCE_BYTES)
SOURCE_SHA256 = hashlib.sha256(SOURCE_BYTES).hexdigest()
RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat()
print(ORIGINAL_NAME, len(SOURCE_BYTES), "bytes  sha256", SOURCE_SHA256)

In [ ]:
#@title 4 · Preflight (observable page evidence + deterministic sampling)
import pymupdf as fitz  # PyMuPDF
doc = fitz.open(str(SOURCE_PATH))
PAGE_COUNT = doc.page_count
if PAGE_COUNT < 1:
    raise SystemExit("PDF_HAS_NO_PAGES")

def sample_pages(n, limit):
    # Deterministic rule: full if small; otherwise first 3 + last 2 + evenly spaced middle, max 12 pages.
    if n <= limit: return list(range(1, n + 1)), "FULL_UNDER_LIMIT"
    picks = set(range(1, min(3, n) + 1)) | set(range(max(1, n - 1), n + 1))
    step = max(1, n // 7)
    picks |= set(range(4, n - 1, step))
    return sorted(picks)[:12], f"FIRST3_LAST2_EVERY_{step}_MAX12"

if FORCE_FULL:
    SAMPLE_PAGES, SAMPLE_RULE = list(range(1, PAGE_COUNT + 1)), "FORCE_FULL"
else:
    SAMPLE_PAGES, SAMPLE_RULE = sample_pages(PAGE_COUNT, FULL_CONVERSION_MAX_PAGES)
FULL_CONVERSION = len(SAMPLE_PAGES) == PAGE_COUNT

PREFLIGHT = []
for pno in SAMPLE_PAGES:
    page = doc[pno - 1]
    text_chars = len(page.get_text("text").strip())
    image_count = len(page.get_images(full=True))
    drawing_count = len(page.get_drawings())
    try:
        table_count = len(page.find_tables().tables)
    except Exception:
        table_count = 0
    PREFLIGHT.append({
        "page": pno,
        "text_chars": text_chars,
        "image_count": image_count,
        "vector_drawing_count": drawing_count,
        "table_candidates": table_count,
        # scanned-page evidence: almost no extractable text AND raster image present
        "needs_ocr_by_text_density": text_chars < MIN_TEXT_CHARS_PER_PAGE and image_count > 0,
    })
doc.close()
OCR_REQUIRED = any(p["needs_ocr_by_text_density"] for p in PREFLIGHT)
print(f"pages={PAGE_COUNT} sample={SAMPLE_PAGES} rule={SAMPLE_RULE} full={FULL_CONVERSION} ocr_required={OCR_REQUIRED}")
for p in PREFLIGHT: print(p)

In [ ]:
#@title 5 · Run Marker (OCR/Surya only if preflight says so; no LLM)
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.config.parser import ConfigParser
from marker.output import text_from_rendered

config = {
    "output_format": "markdown",
    "page_range": ",".join(str(p - 1) for p in SAMPLE_PAGES),  # Marker is 0-indexed
    "use_llm": False,
    "force_ocr": False,
    "disable_ocr": not OCR_REQUIRED,   # free/default text extraction first
}
parser = ConfigParser(config)
t0 = time.time()
converter = PdfConverter(
    config=parser.generate_config_dict(),
    artifact_dict=create_model_dict(),
    processor_list=parser.get_processors(),
    renderer=parser.get_renderer(),
)
rendered = converter(str(SOURCE_PATH))
MARKDOWN, _, IMAGES = text_from_rendered(rendered)
MARKER_SECONDS = round(time.time() - t0, 2)
if not MARKDOWN.strip():
    raise SystemExit("MARKER_PRODUCED_EMPTY_MARKDOWN")

# Per-page OCR evidence as reported by Marker itself (pdftext vs surya).
page_stats = (getattr(rendered, "metadata", {}) or {}).get("page_stats", []) or []
OCR_BY_PAGE = {}
for st in page_stats:
    pid = st.get("page_id")
    if pid is not None:
        OCR_BY_PAGE[int(pid) + 1] = st.get("text_extraction_method") == "surya"
print(f"marker done in {MARKER_SECONDS}s, markdown chars={len(MARKDOWN)}, images={len(IMAGES)}, ocr pages={[p for p,v in OCR_BY_PAGE.items() if v]}")

In [ ]:
#@title 6 · Write bundle, hashes, flags → download ZIP
import json, zipfile, re, platform
md_path = WORK / "output" / "document.md"; md_path.write_text(MARKDOWN, encoding="utf-8")
outputs = []
def add_output(rel, data):
    outputs.append({"path": rel, "sha256": hashlib.sha256(data).hexdigest(), "size_bytes": len(data)})
add_output("output/document.md", md_path.read_bytes())
for name, img in IMAGES.items():
    safe = re.sub(r"[^A-Za-z0-9._-]", "_", name)
    p = WORK / "output" / "images" / safe
    img.save(p) if hasattr(img, "save") else p.write_bytes(img)
    add_output(f"output/images/{safe}", p.read_bytes())

pages_flags = []
for pf in PREFLIGHT:
    ocr = OCR_BY_PAGE.get(pf["page"], False)
    diagram = pf["vector_drawing_count"] >= 20 or pf["image_count"] > 0 and pf["text_chars"] < 200
    table = pf["table_candidates"] > 0
    pages_flags.append({
        "page": pf["page"],
        "has_images": pf["image_count"] > 0,
        "possible_diagram": bool(diagram),
        "possible_table": bool(table),
        "ocr_used": bool(ocr),
        "needs_visual_qa": bool(diagram or table or ocr or pf["image_count"] > 0),
        "evidence": {k: pf[k] for k in ("text_chars", "image_count", "vector_drawing_count", "table_candidates")},
    })
OCR_USED = any(p["ocr_used"] for p in pages_flags)
now = datetime.datetime.now(datetime.timezone.utc).isoformat()

manifest = {
    "schema": BUNDLE_SCHEMA, "capability_id": CAPABILITY_ID, "created_at": now,
    "source": {"original_filename": ORIGINAL_NAME, "stored_path": "source/original.pdf",
               "sha256": SOURCE_SHA256, "size_bytes": len(SOURCE_BYTES), "page_count": PAGE_COUNT},
    "marker": {"version": MARKER_VERSION, "config": config},
    "environment": {"runtime": "google-colab-interactive", "gpu": gpu_name, "torch": torch.__version__,
                    "python": platform.python_version(), "paid_resources": False},
    "sampling": {"full_conversion": FULL_CONVERSION, "sample_pages": SAMPLE_PAGES, "rule": SAMPLE_RULE,
                 "full_conversion_max_pages": FULL_CONVERSION_MAX_PAGES},
    "ocr_used": OCR_USED, "vlm_used": False, "external_llm_used": False,
    "outputs": outputs,
    "note": "Markdown is derived output. source/original.pdf is the archival artifact.",
}
quality = {
    "markdown_nonempty": True, "markdown_chars": len(MARKDOWN),
    "markdown_lines": MARKDOWN.count("\n") + 1,
    "heading_count": len(re.findall(r"^#+ ", MARKDOWN, re.M)),
    "table_row_count": len(re.findall(r"^\|.*\|\s*$", MARKDOWN, re.M)),
    "image_refs_in_markdown": len(re.findall(r"!\[[^\]]*\]\(", MARKDOWN)),
    "images_extracted": len(IMAGES),
    "pages_converted": len(SAMPLE_PAGES), "pages_total": PAGE_COUNT,
    "pages_needing_visual_qa": [p["page"] for p in pages_flags if p["needs_visual_qa"]],
    "semantic_review_performed": False,
}
page_flags = {"schema": BUNDLE_SCHEMA, "basis": "PyMuPDF preflight counts + Marker page_stats; no semantic interpretation", "pages": pages_flags}
run = {"capability_id": CAPABILITY_ID, "status": "PASS", "started_at": RUN_STARTED, "finished_at": now,
       "gpu_used": True, "gpu": gpu_name, "marker_seconds": MARKER_SECONDS, "paid_spend": 0,
       "credentials_used": False, "executor": "OWNER_INTERACTIVE_COLAB"}
for name, obj in (("manifest.json", manifest), ("quality.json", quality), ("page_flags.json", page_flags), ("run.json", run)):
    (WORK / name).write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")

stem = re.sub(r"[^A-Za-z0-9._-]", "_", pathlib.Path(ORIGINAL_NAME).stem)
ZIP = WORK.parent / f"p2p-marker-{stem}-{SOURCE_SHA256[:12]}.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(WORK.rglob("*")):
        if f.is_file(): zf.write(f, f.relative_to(WORK).as_posix())
print("bundle:", ZIP, ZIP.stat().st_size, "bytes")
print("Next: validate with  python .p2p/system/p2p_capabilities/marker_bundle_validator.py", ZIP.name)
files.download(str(ZIP))